In [2]:
#!/usr/bin/env python3
import os
import re

INPUT_DIR = r"test_data\Smith\TSPrd(time)\Solomon\50"
OUTPUT_DIR = os.path.join(INPUT_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def parse_vertices_block(lines):
    """
    Tìm dòng <VERTICES> và đọc toàn bộ dữ liệu sau đó.
    Mỗi dòng dữ liệu có ít nhất 7 trường:
      X  Y  DEMAND  OPEN  CLOSE  SERVICE  RELEASE
    Trả về list các bản ghi [x, y, demand, release]
    """
    start = None
    for i, line in enumerate(lines):
        if line.strip().startswith("<VERTICES>"):
            start = i + 1
            break
    if start is None:
        return []

    data = []
    number_pattern = re.compile(r"[-+]?\d+(\.\d+)?")

    for raw in lines[start:]:
        line = raw.strip()
        if not line:
            continue
        # Bỏ các dòng metadata khác nếu có
        if line.startswith("<") and line.endswith(">"):
            break

        # Tách theo whitespace
        parts = line.split()
        # Một số file có thể có dấu tab/khoảng trắng lẫn lộn — xử lý chung
        # Yêu cầu tối thiểu 7 giá trị số (X, Y, DEMAND, OPEN, CLOSE, SERVICE, RELEASE)
        # Nếu không đủ thì bỏ qua dòng
        nums = []
        for p in parts:
            # bỏ các ký tự lạ, giữ số
            if number_pattern.match(p):
                nums.append(p)
        if len(nums) < 7:
            # nếu không đủ cột, bỏ
            continue

        # Lấy đúng 7 trường đầu
        x, y, demand, _open, _close, _service, release = nums[:7]

        # Ép về số (int nếu được, float nếu cần)
        def to_number(s):
            return int(float(s)) if re.match(r"^-?\d+(\.0+)?$", s) else float(s)

        xv = to_number(x)
        yv = to_number(y)
        dv = to_number(demand)
        rv = to_number(release)

        data.append([xv, yv, dv, rv])

    return data

def force_demands(rows):
    """Đặt DEMAND=0 cho dòng đầu, các dòng còn lại DEMAND=1."""
    if not rows:
        return rows
    rows[0][2] = 0
    for i in range(1, len(rows)):
        rows[i][2] = 1
    return rows

def write_dat(path, rows):
    """Ghi file .dat mới chỉ gồm 4 cột, tab-separated, kèm header."""
    with open(path, "w", encoding="utf-8") as f:
        f.write("XCOORD\tYCOORD\tDEMAND\tRELEASE_DATE\n")
        for x, y, d, r in rows:
            f.write(f"{x}\t{y}\t{d}\t{r}\n")

def process_file(in_path, out_path):
    with open(in_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    rows = parse_vertices_block(lines)
    rows = force_demands(rows)
    write_dat(out_path, rows)

def main():
    count_in, count_out = 0, 0
    for name in os.listdir(INPUT_DIR):
        if not name.lower().endswith(".dat"):
            continue
        in_path = os.path.join(INPUT_DIR, name)
        out_path = os.path.join(OUTPUT_DIR, name)
        count_in += 1
        try:
            process_file(in_path, out_path)
            count_out += 1
            print(f"OK  -> {name}  => processed/{name}")
        except Exception as e:
            print(f"FAIL -> {name}: {e}")
    print(f"\nDone. {count_out}/{count_in} file(s) processed. Output: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()


OK  -> C101_0.5.dat  => processed/C101_0.5.dat
OK  -> C101_1.5.dat  => processed/C101_1.5.dat
OK  -> C101_1.dat  => processed/C101_1.dat
OK  -> C101_2.5.dat  => processed/C101_2.5.dat
OK  -> C101_2.dat  => processed/C101_2.dat
OK  -> C101_3.dat  => processed/C101_3.dat
OK  -> C201_0.5.dat  => processed/C201_0.5.dat
OK  -> C201_1.5.dat  => processed/C201_1.5.dat
OK  -> C201_1.dat  => processed/C201_1.dat
OK  -> C201_2.5.dat  => processed/C201_2.5.dat
OK  -> C201_2.dat  => processed/C201_2.dat
OK  -> C201_3.dat  => processed/C201_3.dat
OK  -> R101_0.5.dat  => processed/R101_0.5.dat
OK  -> R101_1.5.dat  => processed/R101_1.5.dat
OK  -> R101_1.dat  => processed/R101_1.dat
OK  -> R101_2.5.dat  => processed/R101_2.5.dat
OK  -> R101_2.dat  => processed/R101_2.dat
OK  -> R101_3.dat  => processed/R101_3.dat
OK  -> RC101_0.5.dat  => processed/RC101_0.5.dat
OK  -> RC101_1.5.dat  => processed/RC101_1.5.dat
OK  -> RC101_1.dat  => processed/RC101_1.dat
OK  -> RC101_2.5.dat  => processed/RC101_2.5.da